# 02 — Research Pipeline

**Mode:** Live  
**Level:** Fundamentals  
**Estimated time:** 35 minutes

[Watch the original lesson video](https://www.youtube.com/watch?v=u7gib_vrMMQ)

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## What you will build

A live two-stage research pipeline. One model-backed agent extracts factual findings from a bounded source note; another turns those findings into a briefing. The intermediate Spore remains inspectable.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, get_events, require_env, setup_notebook,
    show_callout, show_flow, show_json, show_roles, show_spore,
    show_timeline, stage,
)

setup_notebook(2, 'Research Pipeline')


## Prerequisites and setup

**Course prerequisites:** Complete `course-00-architecture`, `course-01-hello-world`.

**Execution requirements:** Live OpenAI by default. Set `OPENAI_API_KEY` and `PRAVAL_OPENAI_MODEL`. Anthropic is also supported with `PRAVAL_TUTORIAL_PROVIDER=anthropic`, `ANTHROPIC_API_KEY`, and `PRAVAL_ANTHROPIC_MODEL`.

Run the setup cell above first. It configures presentation helpers and a
credential-free lifecycle provider. It does not hide any agent workflow.


## Learning goals

- Select a provider and model explicitly.
- Build a multi-stage pipeline from message types.
- Inspect model output while it moves between agents.
- Use completion tracking instead of sleeps.


## Mental model

A pipeline is a sequence of message contracts, not a chain of direct function calls. The researcher owns source extraction. The writer owns presentation. The Reef preserves their boundary by carrying the intermediate result in a Spore. The model runtime sits inside each agent and translates provider-neutral requests into provider API calls.


In [ ]:
show_flow(
('Request', 'bounded source', 'human'),
('Researcher', 'extract findings', 'agent'),
('Findings Spore', 'visible handoff', 'spore'),
('Writer', 'compose briefing', 'agent'),
)


## Try it

We will now assemble the workflow in small steps. Run each cell, then pause at the inspection that follows it.


### Select the live provider

Configuration is read from the environment. Missing credentials are an explicit error because this lesson certifies real provider behavior.


In [ ]:
import os

provider = os.environ.get("PRAVAL_TUTORIAL_PROVIDER", "openai").lower()
configuration = {
    "openai": ("OPENAI_API_KEY", "PRAVAL_OPENAI_MODEL"),
    "anthropic": ("ANTHROPIC_API_KEY", "PRAVAL_ANTHROPIC_MODEL"),
}
if provider not in configuration:
    raise RuntimeError("PRAVAL_TUTORIAL_PROVIDER must be openai or anthropic")
key_name, model_name = configuration[provider]
values = require_env(key_name, model_name)
model = values[model_name]
show_json({"provider": provider, "model": model}, "Live model selection")


### What just happened?

The provider and model are now explicit runtime inputs; the secret value is never rendered.

### Why this matters

Provider choice belongs in configuration so the same workflow can be tested across supported model families.


### Define the researcher

The researcher calls `chat()` only for extraction, then broadcasts a typed findings payload. Temperature and output length are bounded.


In [ ]:
from praval import agent, broadcast, chat, get_reef, start_agents
from praval.core.reef import reset_reef

reset_reef()
briefings = []
intermediate_spores = []


@agent(
    "course-researcher", provider=provider, model=model,
    responds_to=["research_request"], auto_broadcast=False,
    config={"temperature": 0, "max_output_tokens": 160},
)
def researcher(spore):
    stage("researcher", "model request", "extract three findings", spore)
    findings = chat(
        "Extract exactly three short factual findings from this source:\n"
        + spore.knowledge["source"]
    )
    broadcast({"type": "findings", "text": findings})
    stage("researcher", "broadcast", "findings", spore)


### What just happened?

The agent's model call is scoped to one responsibility. Its externally visible output is the `findings` Spore, not hidden local state.

### Why this matters

Small responsibilities make model prompts easier to evaluate and handoffs easier to inspect.


### Define the writer

The writer listens only for findings, preserves the incoming Spore for inspection, and makes its own bounded provider call.


In [ ]:
@agent(
    "course-writer", provider=provider, model=model,
    responds_to=["findings"], auto_broadcast=False,
    config={"temperature": 0, "max_output_tokens": 120},
)
def writer(spore):
    intermediate_spores.append(spore)
    stage("writer", "model request", "compose briefing", spore)
    briefing = chat(
        "Turn these findings into a two-sentence technical briefing:\n"
        + spore.knowledge["text"]
    )
    briefings.append(briefing)
    stage("writer", "briefing complete", f"{len(briefing)} chars", spore)


### What just happened?

No code directly invokes `writer()`. The Reef schedules it when the research agent publishes a matching Spore.

### Why this matters

Message-driven staging keeps provider calls independently configurable and observable.


### Run the complete pipeline

The source is deliberately short and factual. The certification asserts structure and completion rather than exact natural-language wording.


In [ ]:
source_note = (
    "Praval agents communicate through structured Spores on a Reef. "
    "Handlers filter messages by type. Completion waits include cascading "
    "work triggered by later broadcasts."
)
start_agents(
    researcher,
    writer,
    initial_data={"type": "research_request", "source": source_note},
    channel="course-research",
)
reef = get_reef()
assert reef.wait_for_completion(timeout=180)
assert len(intermediate_spores) == 1
assert len(briefings) == 1 and len(briefings[0].strip()) > 20


### What just happened?

The completion wait covered the first provider call, findings broadcast, second delivery, and second provider call.

### Why this matters

Structural assertions tolerate provider phrasing while still proving the multi-stage behavior occurred.


In [ ]:
show_spore(intermediate_spores[0], "Findings passed to the writer")
show_json({"briefing": briefings[0]}, "Final live briefing")
show_timeline()


## Your turn

Add a third `reviewer` agent that listens for a new `briefing_ready` Spore. Change the writer to broadcast its briefing, and have the reviewer record one quality observation.


In [ ]:
# @agent("course-reviewer", responds_to=["briefing_ready"], auto_broadcast=False)
# def reviewer(spore):
#     review_notes.append({"length": len(spore.knowledge["briefing"])})
# Update writer() to broadcast {"type": "briefing_ready", "briefing": briefing}.


## Common mistake

**Mistake:** Embedding every stage in one large prompt because one provider call is shorter code.

**Correction:** Keep distinct responsibilities when you need inspectable handoffs, separate failure handling, or provider choices per stage.


<details>
<summary>Under the hood</summary>

`chat()` uses the currently executing agent's model runtime. The resulting text is ordinary data until the handler places it in a Spore. Reef completion accounting includes later broadcasts, so the caller gets a true pipeline completion boundary.

</details>


## Recap

- A pipeline is expressed through typed message handoffs.
- Model calls remain inside focused agent responsibilities.
- Intermediate outputs can be inspected as real Spores.
- Live assertions should test structure and behavior, not exact prose.


## Cleanup

Release every agent, transport, temporary file, or external resource created by this lesson. Cleanup is part of the example, not an afterthought.


In [ ]:
researcher._praval_agent.close()
writer._praval_agent.close()
assert reef.shutdown(timeout=10)
stage("reef", "shutdown", "live agents and channels closed")
show_timeline()


### Next lesson

Continue to `03_feedback_loop.ipynb` to build a pipeline that can send work backward for revision and still terminate predictably.
